# **Artificial Neural Networks and Deep Learning**

---

## **Lecture 2: Overfitting and Regularisation**

<img src="https://drive.google.com/uc?export=view&id=17CdxwTrnLDwP43PC_K2hT05RLs1noo1_" width="500"/>



### 🌐 **Google Drive Connection**

In [ ]:
from google.colab import drive
drive.mount("/gdrive")
current_dir = "/gdrive/My\\ Drive/[2026-2027]\\ AN2DL/Lecture\\ 2"
%cd $current_dir

### ⚙️ **Libraries Import**

In [ ]:
# Install the libraries that are not preinstalled on Colab
!pip install -q torchinfo

# Set seed for reproducibility
SEED = 42

import os
import copy
import re
import json
import random
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary

# Trained weights are saved in this folder, one file per model
MODELS_DIR = 'models/Overfitting and Regularisation'
os.makedirs(MODELS_DIR, exist_ok=True)


def weights_path(name):
    """File where the weights of the model called `name` are saved."""
    return f"{MODELS_DIR}/{re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()}.pt"


# Seed the Python, NumPy and PyTorch random number generators
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use the GPU when available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Shared plot style: colour-blind-safe palette, light grid
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
sns.set_theme(style='whitegrid', palette=PALETTE, font_scale=1.2,
              rc={'grid.color': '#e1e0d9', 'axes.edgecolor': '#c3c2b7', 'lines.linewidth': 2})

### ⏳ **Data Loading**

The Reuters newswire dataset, in the version distributed with Keras: 11,228 short news articles published by Reuters in 1987, each labelled with one of 46 topics, from company earnings to cocoa. The words are already turned into integers: each word is replaced by its rank by frequency in the whole collection (1 is the most frequent word).

In [ ]:
# Download and unzip the dataset
DATASET_NAME = "reuters.zip"
DATASET_ID = "18a2QOZ15UwQs7pVgvhSl-bAP-Qd4M5F7"
if not os.path.exists(DATASET_NAME):
    !gdown -q {DATASET_ID} -O {DATASET_NAME}
    !unzip -q -o {DATASET_NAME}

# Articles as lists of word ranks, labels as topic indices from 0 to 45
data = np.load('reuters.npz', allow_pickle=True)
sequences, labels = data['x'], data['y']
print(f"Articles: {len(sequences)}, labels: {labels.shape}")

### 🔎 **Exploration and Data Analysis**

In [ ]:
# Names of the 46 topics: the dataset stores only their indices
class_names = ['cocoa', 'grain', 'veg-oil', 'earn', 'acq', 'wheat', 'copper', 'housing', 'money-supply',
               'coffee', 'sugar', 'trade', 'reserves', 'ship', 'cotton', 'carcass', 'crude', 'nat-gas',
               'cpi', 'money-fx', 'interest', 'gnp', 'meal-feed', 'alum', 'oilseed', 'gold', 'tin',
               'strategic-metal', 'livestock', 'retail', 'ipi', 'iron-steel', 'rubber', 'heat', 'jobs',
               'lei', 'bop', 'zinc', 'orange', 'pet-chem', 'dlr', 'gas', 'silver', 'wpi', 'hog', 'lead']
num_classes = len(class_names)

# Map each rank back to its word, to read the articles
with open('reuters_word_index.json') as f:
    index_word = {rank: word for word, rank in json.load(f).items()}

# The beginning of three articles: cocoa, company earnings (earn) and acquisitions (acq)
for i in [0, 8, 4]:
    text = ' '.join(index_word[rank] for rank in sequences[i][:50])
    print(f"Topic: {class_names[labels[i]]} ({len(sequences[i])} words)")
    print(textwrap.fill(text, 100) + ' ...\n')

In [ ]:
# Number of articles per topic, from the largest to the smallest
counts = np.bincount(labels, minlength=num_classes)
order = np.argsort(counts)[::-1]

fig, ax = plt.subplots(figsize=(16, 4.5))
ax.bar([class_names[k] for k in order], counts[order], color=PALETTE[0])
ax.set(title='Articles per topic', ylabel='Articles')
ax.tick_params(axis='x', rotation=90)
ax.grid(axis='x', visible=False)
ax.margins(x=0.01)
plt.tight_layout()
plt.show()

smallest = [class_names[k] for k in np.flatnonzero(counts == counts.min())]
print(f"Largest topic: {class_names[order[0]]}, {counts.max()} articles ({counts.max() / len(labels):.0%})")
print(f"Smallest topics: {', '.join(smallest)}, {counts.min()} articles each")
print(f"Imbalance ratio: {counts.max() / counts.min():.0f} to 1")

### 📚 **Classification Metrics**


**Binary Classification Metrics**

<img src="https://drive.google.com/uc?export=view&id=1hsY61MB2IW2yE815RY9nLZz5jp9RJk7y" width="300"/>

$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$

$\text{Precision} = \frac{TP}{TP + FP}$

$\text{Recall} = \frac{TP}{TP + FN}$

$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$

&nbsp;

<img src="https://drive.google.com/uc?export=view&id=1YiS5WGpl3YQoA8Wa7rXDcTEhDLzCJtPn" width="400"/>

&nbsp;

---

**Multiclass Classification Metrics**

With $K$ classes, $N_i$ samples in class $i$ and $N$ samples in total:

<img src="https://drive.google.com/uc?export=view&id=1nipRDCRWYSwXKkmYUoFOiAOlSwAul3I_" width="400"/>

$\text{Accuracy} = \frac{1}{N} \sum_{i=1}^{K} TP_i$

$\text{Precision}_i = \frac{TP_i}{TP_i + FP_i}$

$\text{Recall}_i = \frac{TP_i}{TP_i + FN_i}$

$F1_i = 2 \cdot \frac{\text{Precision}_i \cdot \text{Recall}_i}{\text{Precision}_i + \text{Recall}_i}$


&nbsp;

**Macro Averaging**: every class counts the same

$\text{Precision}_{macro} = \frac{1}{K} \sum_{i=1}^{K} \text{Precision}_i$

$\text{Recall}_{macro} = \frac{1}{K} \sum_{i=1}^{K} \text{Recall}_i$

$F1_{macro} = \frac{1}{K} \sum_{i=1}^{K} F1_i$

&nbsp;

**Weighted Averaging**: every class counts as much as its share of samples

$\text{Precision}_{weighted} = \sum_{i=1}^{K} \frac{N_i}{N} \, \text{Precision}_i$

$\text{Recall}_{weighted} = \sum_{i=1}^{K} \frac{N_i}{N} \, \text{Recall}_i$

$F1_{weighted} = \sum_{i=1}^{K} \frac{N_i}{N} \, F1_i$

&nbsp;

With unbalanced classes the macro average is the stricter one: a model that ignores the small classes still gets a high accuracy and a high weighted F1, but a low macro F1. Reuters is heavily unbalanced, so let us see what the metrics say about a classifier that ignores the articles altogether.

In [ ]:
# A useless classifier: whatever the article, it answers the most frequent topic
majority = np.full_like(labels, counts.argmax())
print(f"Accuracy:            {accuracy_score(labels, majority):.4f}")
print(f"F1 score (weighted): {f1_score(labels, majority, average='weighted', zero_division=0):.4f}")
print(f"F1 score (macro):    {f1_score(labels, majority, average='macro', zero_division=0):.4f}")

Accuracy gives this classifier credit for the largest topic alone, the weighted F1 score still a little; the macro F1 score exposes it, since it scores 0 on 45 topics out of 46. The macro F1 score is the metric we monitor in this notebook.

### 🔄 **Data Preprocessing**

In [ ]:
# Keep the 2,000 most frequent words: each article becomes a vector with a 1 for each of these words it
# contains and a 0 for the others (multi-hot encoding), so word order and repetitions are lost
VOCAB_SIZE = 2000
X = np.zeros((len(sequences), VOCAB_SIZE), dtype=np.float32)
for i, sequence in enumerate(sequences):
    ranks = np.array(sequence)
    X[i, ranks[ranks <= VOCAB_SIZE] - 1] = 1.0   # rank 1 goes to column 0
y = labels.astype(np.int64)
input_features = VOCAB_SIZE

print(f"Features: {X.shape}, labels: {y.shape}")
print(f"Distinct words per article, among the {VOCAB_SIZE} kept: {X.sum(axis=1).mean():.0f} on average")

In [ ]:
# Hold out 15% of the articles for the final test and as many for validation, keeping the topic proportions
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=len(y_test), random_state=SEED, stratify=y_train_val
)

print(f"Training set:   {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set:       {X_test.shape}, {y_test.shape}")

In [ ]:
# Pair features with labels in PyTorch datasets
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

# Number of samples processed before each weight update
BATCH_SIZE = 32

# Shuffle only the training data; validation and test keep a fixed order
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [ ]:
# Inspect one training batch
xb, yb = next(iter(train_loader))
print(f"Features batch shape: {xb.shape}")
print(f"Labels batch shape: {yb.shape}")

### 🛠️ **Model Building**

In [ ]:
class FeedForwardNet(nn.Module):
    """Fully connected network: `hidden_layers` layers of `hidden_size` neurons with ReLU and optional dropout."""

    def __init__(self, in_features, hidden_layers, hidden_size, num_classes, dropout_rate=0.0):
        super().__init__()
        layers, width = [], in_features
        for _ in range(hidden_layers):
            layers += [nn.Linear(width, hidden_size), nn.ReLU()]
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            width = hidden_size

        # Output layer: one logit per class
        layers.append(nn.Linear(width, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
# A deliberately large network (about 1.6 million parameters) for about 7,900 training articles, so that it can overfit
HIDDEN_LAYERS = 3
HIDDEN_SIZE = 512


def make_model(dropout_rate=0.0):
    """Build a fresh network; reseeding gives every experiment the same initial weights."""
    torch.manual_seed(SEED)
    return FeedForwardNet(input_features, HIDDEN_LAYERS, HIDDEN_SIZE, num_classes, dropout_rate).to(device)


summary(make_model(), input_size=(BATCH_SIZE, input_features))

### 🧮 **Hyperparameters**

In [ ]:
LEARNING_RATE = 1e-3
EPOCHS = 100

# Metric monitored on the validation set: the topics are unbalanced, so the macro F1 score
METRIC = 'f1'

# Epochs without improvement of the validation metric before early stopping
PATIENCE = 20

criterion = nn.CrossEntropyLoss()

### 🧠 **Model Training**

The training functions of Lecture 1 grow three features:
- the **metric** to monitor, here the macro F1 score;
- **L1 and L2 penalties** added to the loss that the optimiser minimises;
- **early stopping**: stop when the validation metric has not improved for `patience` epochs, then restore the weights of the best epoch.

The plotting and evaluation helpers are the same as in Lecture 1, collapsed in the next cell. `fit` can still record the weights and the gradients for the histograms of Lecture 1 (`track_parameters`); here it is off, and the 📈 section compares the weights of the final models instead.

In [ ]:
# @title Monitoring and evaluation helpers

def plot_history(history, max_epochs=None):
    """Plot training and validation curves of the loss and of the metric, marking the best epoch."""
    titles = {'loss': 'Loss', 'accuracy': 'Accuracy', 'f1': 'F1 score (macro)'}
    metrics = [key.removeprefix('train_') for key in history if key.startswith('train_')]
    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4.5))
    epochs = range(1, len(history['train_loss']) + 1)

    for ax, metric in zip(axes, metrics):
        ax.plot(epochs, history[f'train_{metric}'], color=PALETTE[0], label='Training')
        ax.plot(epochs, history[f'val_{metric}'], color=PALETTE[1], label='Validation')
        if 'best_epoch' in history:
            ax.axvline(history['best_epoch'], color='#898781', linestyle='--', linewidth=1, label='Best epoch')
        ax.set(title=titles.get(metric, metric), xlabel='Epoch', xlim=(1, max_epochs or len(epochs)))
        ax.legend()

    plt.tight_layout()
    plt.show()


def record_parameters(model, record, epoch, max_values=10_000):
    """Append the output layer's weights and bias and the gradients of every layer (at most `max_values` each)."""
    layers = [module for module in model.modules() if isinstance(module, nn.Linear)]

    def sample(tensor):
        # Copy, so that later in-place weight updates do not change the stored values
        tensor = tensor.detach().flatten()
        return tensor[::max(1, tensor.numel() // max_values)].cpu().numpy().copy()

    record['epochs'].append(epoch)
    for name, param in (('Output layer weights', layers[-1].weight), ('Output layer bias', layers[-1].bias)):
        record['values'].setdefault(name, []).append(sample(param))
    for i, layer in enumerate(layers, start=1):
        grad = torch.cat([layer.weight.grad.flatten(), layer.bias.grad.flatten()])
        record['gradients'].setdefault(f'Layer {i}', []).append(sample(grad))


def plot_histograms(tensors, epochs, kind='values'):
    """TensorBoard-style histograms of each tensor over the recorded epochs, the oldest at the back.

    For gradients, exact zeros are left out and the axis covers the central 98% of the values.
    """
    ramp = ['#cde2fb', '#0d366b'] if kind == 'values' else ['#fbd6c4', '#7a2a0a']
    colours = sns.blend_palette(ramp, as_cmap=True)
    kernel = np.exp(-0.5 * np.linspace(-2, 2, 9) ** 2)
    fig, axes = plt.subplots(1, len(tensors), figsize=(4.5 * len(tensors), 5), squeeze=False)

    for ax, (name, slices) in zip(axes[0], tensors.items()):
        if kind == 'gradients':
            # Exact zeros (neurons inactive on the batch) would dwarf the rest: leave them out
            slices = [values[values != 0] for values in slices]
        pooled = np.concatenate(slices)
        n = len(slices)

        # Values: the full range; gradients: the central 98%, so that a few outliers do not squeeze the plot.
        # The range is widened by 15% on each side so the smoothed tails are not clipped
        low, high = (pooled.min(), pooled.max()) if kind == 'values' else np.percentile(pooled, [1, 99])
        margin = 0.15 * (high - low) + 1e-8
        bins = np.linspace(low - margin, high + margin, 81)
        centres = (bins[:-1] + bins[1:]) / 2

        # Histogram of each slice, smoothed with a small Gaussian into a curve
        counts = np.stack([np.convolve(np.histogram(values, bins)[0], kernel, mode='same') for values in slices])
        heights = 5 * counts / max(counts.max(), 1)

        # One ridge per slice, from the oldest (top, back) to the newest (bottom, front)
        for k, height in enumerate(heights):
            ax.fill_between(centres, n - 1 - k, n - 1 - k + height, facecolor=colours(k / max(1, n - 1)),
                            edgecolor='white', linewidth=0.6, zorder=k)

        ticks = range(0, n, max(1, round(n / 5)))   # about five evenly spaced epoch labels
        ax.set_yticks([n - 1 - k for k in ticks], [epochs[k] for k in ticks])
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position('right')
        ax.set(title=name, xlabel='Value' if kind == 'values' else 'Gradient', ylabel='Epoch',
               xlim=(bins[0], bins[-1]))
        # A few ticks in scientific notation, so that small gradients stay readable
        ax.xaxis.set_major_locator(plt.MaxNLocator(5))
        ax.ticklabel_format(axis='x', style='sci', scilimits=(-2, 3))
        ax.grid(False)

    plt.tight_layout()
    plt.show()


def plot_parameters(history):
    """Histograms of the output layer's weights and bias, then of the gradients of every layer."""
    record = history['parameters']
    plot_histograms(record['values'], record['epochs'], kind='values')
    plot_histograms(record['gradients'], record['epochs'], kind='gradients')


def predict(model, loader, device):
    """Return true and predicted labels for every sample in a loader."""
    model.eval()
    targets, preds = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.append(model(x.to(device)).argmax(dim=1).cpu())
            targets.append(y)
    return torch.cat(targets).numpy(), torch.cat(preds).numpy()


def print_metrics(y_true, y_pred):
    """Print accuracy, the macro precision, recall and F1 score, and the weighted F1 score."""
    print(f"Accuracy:            {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision (macro):   {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Recall (macro):      {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 score (macro):    {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 score (weighted): {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")


def plot_confusion_matrix(y_true, y_pred, class_names, top=None, normalize=False, title='Confusion matrix'):
    """Plot a confusion matrix: rows are true classes, columns predicted classes.

    With `top`, keep the `top` largest true classes and merge all the others into 'Other'
    (so the Other/Other cell also counts mix-ups among the merged classes). With `normalize`,
    each row sums to 1 and the diagonal holds the recall of each class.
    """
    if top is not None and top < len(class_names):
        largest = np.argsort(np.bincount(y_true, minlength=len(class_names)), kind='stable')[::-1][:top]
        merge = np.full(len(class_names), top)
        merge[largest] = np.arange(top)
        y_true, y_pred = merge[y_true], merge[y_pred]
        class_names = [class_names[k] for k in largest] + ['Other']
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)),
                          normalize='true' if normalize else None)
    size = 2 + min(1.1, 7 / len(class_names)) * len(class_names)
    fig, ax = plt.subplots(figsize=(size + 1, size))
    sns.heatmap(cm, annot=True, fmt='.2f' if normalize else 'd', cmap='Blues', cbar=False,
                square=True, xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set(xlabel='Predicted label', ylabel='True label', title=title)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
def score(y_true, y_pred, metric):
    """Monitored metric: 'accuracy', or 'f1' for the F1 score averaged over the classes (macro)."""
    if metric == 'accuracy':
        return accuracy_score(y_true, y_pred)
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, metric, l1_lambda=0.0, l2_lambda=0.0):
    """Run one training epoch; return mean loss (without penalties) and the metric."""
    model.train()
    total_loss, targets, preds = 0.0, [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Forward pass: predictions and loss
        logits = model(x)
        loss = criterion(logits, y)

        # L1 and L2 penalties on all the parameters, added to the loss being minimised
        penalty = 0.0
        if l1_lambda > 0:
            penalty += l1_lambda * sum(p.abs().sum() for p in model.parameters())
        if l2_lambda > 0:
            penalty += l2_lambda * sum(p.pow(2).sum() for p in model.parameters())

        # Backward pass: gradients, then weight update
        optimizer.zero_grad()
        (loss + penalty).backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        targets.append(y.cpu())
        preds.append(logits.argmax(dim=1).cpu())

    return total_loss / len(loader.dataset), score(torch.cat(targets), torch.cat(preds), metric)

In [ ]:
def evaluate(model, loader, criterion, device, metric):
    """Return mean loss and the metric on a loader, without updating the weights."""
    model.eval()
    total_loss, targets, preds = 0.0, [], []

    # No gradients are needed for evaluation
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item() * x.size(0)
            targets.append(y.cpu())
            preds.append(logits.argmax(dim=1).cpu())

    return total_loss / len(loader.dataset), score(torch.cat(targets), torch.cat(preds), metric)

In [ ]:
def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs, metric='f1',
        patience=0, l1_lambda=0.0, l2_lambda=0.0, verbose=10, live=0, track_parameters=0):
    """Train for up to `epochs` epochs and return the history.

    With patience > 0, stop after `patience` epochs without improvement of the validation
    metric and restore the weights of the best epoch. Monitoring options: a log line every
    `verbose` epochs, live plots every `live` epochs, parameters recorded every
    `track_parameters` epochs (0 switches each of them off).
    """
    history = {'train_loss': [], 'val_loss': [], f'train_{metric}': [], f'val_{metric}': []}
    if track_parameters:
        history['parameters'] = {'epochs': [], 'values': {}, 'gradients': {}}
    best_score, best_state, wait = -np.inf, None, 0
    log = []

    for epoch in range(1, epochs + 1):
        train_loss, train_score = train_one_epoch(model, train_loader, criterion, optimizer, device,
                                                  metric, l1_lambda, l2_lambda)
        val_loss, val_score = evaluate(model, val_loader, criterion, device, metric)
        for key, value in zip(['train_loss', 'val_loss', f'train_{metric}', f'val_{metric}'],
                              [train_loss, val_loss, train_score, val_score]):
            history[key].append(value)
        if track_parameters and (epoch == 1 or epoch % track_parameters == 0):
            record_parameters(model, history['parameters'], epoch)

        # Keep a copy of the best weights seen so far
        if val_score > best_score:
            best_score, wait = val_score, 0
            history['best_epoch'] = epoch
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
        stop = patience > 0 and wait >= patience

        # Which of the log line and the live plots are due at this epoch
        last = epoch == epochs or stop
        line = (f"Epoch {epoch:3d}/{epochs} | Train: loss {train_loss:.4f}, {metric} {train_score:.4f} | "
                f"Val: loss {val_loss:.4f}, {metric} {val_score:.4f}")
        log_now = verbose > 0 and (epoch == 1 or epoch % verbose == 0 or last)
        live_now = live > 0 and (epoch == 1 or epoch % live == 0 or last)
        if log_now:
            log.append(line)
            if not live:
                print(line)

        # Redraw the plots in place, with the log below ending on the current epoch
        if live_now:
            clear_output(wait=True)
            plot_history(history, max_epochs=epochs)
            print('\n'.join(log if log_now else log + [line]))
        if stop:
            print(f"Early stopping at epoch {epoch}")
            break

    # With early stopping, go back to the weights of the best epoch
    if patience > 0:
        model.load_state_dict(best_state)
        print(f"Best epoch {history['best_epoch']} | val {metric} {best_score:.4f} (weights restored)")
    else:
        print(f"Best epoch {history['best_epoch']} | val {metric} {best_score:.4f} (weights of the last epoch kept)")
    return history

In [ ]:
# Trained models and their histories, one entry per experiment
models, histories = {}, {}

#### **Experiment 1: Baseline**

No regularisation and no early stopping: the network trains for all the epochs and keeps the weights of the last one. Watch the gap between training and validation.

In [ ]:
%%time
model = make_model()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, metric=METRIC,
              verbose=10, live=5)
models['Baseline'], histories['Baseline'] = model, history
torch.save(model.state_dict(), weights_path('Baseline'))

In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, top=10, normalize=True,
                      title='Confusion matrix (validation set)')

#### **Bias and Variance Tradeoff**

A case of parallel universes:

<img src="https://assets.insightmediagroup.io/media/wp-content/uploads/2020/09/1BVhTW4c1FPFa41ReTqIBXQ.jpeg" width=400px>

- **Bias** measures the systematic error introduced by a model's assumptions. High bias leads to **underfitting**, as the model fails to capture the underlying patterns.
- **Variance** measures how much the model's predictions change across different training sets. High variance leads to **overfitting**, as the model becomes too sensitive to the training data.

How to tune model complexity:

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20251212100155481380/70830267-.webp" width=600px>

As model complexity increases, **bias decreases** while **variance increases**. The goal is to find the optimal complexity that minimises the total generalisation error, balancing underfitting and overfitting.

#### **Experiment 2: Early Stopping**

Same network, but training stops after `PATIENCE` epochs without improvement of the validation F1 score, and the weights of the best epoch are restored. Every experiment starts from the same initial weights, so this run differs from the baseline only in the stopping rule.

In [ ]:
%%time
model = make_model()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, metric=METRIC,
              patience=PATIENCE, verbose=10, live=5)
models['Early stopping'], histories['Early stopping'] = model, history
torch.save(model.state_dict(), weights_path('Early stopping'))

In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, top=10, normalize=True,
                      title='Confusion matrix (validation set)')

<img src="https://drive.google.com/uc?export=view&id=1wBEbGsDgGGliPVYZW_SqH-xCwb9VpLG8" width="500"/>

#### **Experiment 3: Early Stopping and Dropout**

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/6/66/Neural_Network_Dropout.svg/1280px-Neural_Network_Dropout.svg.png" width="600"/>

During training, dropout switches off each hidden neuron with probability `DROPOUT_RATE`, so the network cannot rely on any single neuron. At evaluation time all neurons are active.

In [ ]:
# Probability of dropping each hidden neuron during training
DROPOUT_RATE = 0.5

# Dropout layers appear after every hidden ReLU
summary(make_model(DROPOUT_RATE), input_size=(BATCH_SIZE, input_features))

In [ ]:
%%time
model = make_model(DROPOUT_RATE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, metric=METRIC,
              patience=PATIENCE, verbose=10, live=5)
models['Dropout'], histories['Dropout'] = model, history
torch.save(model.state_dict(), weights_path('Dropout'))

In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, top=10, normalize=True,
                      title='Confusion matrix (validation set)')

#### **Experiment 4: Early Stopping and L2 Regularisation**

$$\mathcal{L}_{\mathrm{reg}} (y, \hat{y}, w) = \mathcal{L}(y, \hat{y}) + \lambda \cdot R(w)$$

where:
- $\mathcal{L}(y, \hat{y})$ is a generic loss function measuring the discrepancy between predictions $\hat{y}$ and ground truth $y$
- $R(w)$ is the regularisation term on the weights:
  - **L2 (Ridge)**: $R(w) = ||w||_2^2 = \sum^K_{k=1}w_k^2$
  - **L1 (Lasso)**: $R(w) = ||w||_1 = \sum^K_{k=1}|w_k|$
  - **Elastic Net**: $R(w) = \alpha||w||_1 + \beta||w||_2^2$
- $\lambda \geq 0$ is the regularisation coefficient controlling the trade-off between loss minimisation and weight regularisation

In [ ]:
# Coefficient of the L2 penalty
L2_LAMBDA = 1e-4

In [ ]:
%%time
model = make_model()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, metric=METRIC,
              patience=PATIENCE, l2_lambda=L2_LAMBDA, verbose=10, live=5)
models['L2'], histories['L2'] = model, history
torch.save(model.state_dict(), weights_path('L2'))

In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, top=10, normalize=True,
                      title='Confusion matrix (validation set)')

#### **Experiment 5: Early Stopping and AdamW**

With Adam, the L2 regularisation term is added to the loss function, affecting the gradients:

$$\mathcal{L}_{\text{total}} = \mathcal{L}(y, \hat{y}) + \lambda||w||_2^2$$

$$\nabla_w \mathcal{L}_{\text{total}} = \nabla_w \mathcal{L}(y, \hat{y}) + 2\lambda w$$

Update rule:
$$w_{t+1} = w_t - \alpha \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}$$

where $m_t$ and $v_t$ are computed using gradients that **include** the L2 term.

**AdamW (Decoupled Weight Decay)**

The weight decay is applied **directly** to the weights, separate from the adaptive gradient computation:

$$\nabla_w \mathcal{L} = \nabla_w \mathcal{L}(y, \hat{y})$$

Update rule (as in `torch.optim.AdamW`, where the decay is scaled by the learning rate $\alpha$):
$$w_{t+1} = (1 - \alpha\lambda)w_t - \alpha \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}$$

where $m_t$ and $v_t$ are computed using gradients **without** the L2 term.

**Note**: In AdamW, the weight decay $\lambda$ acts as a true regulariser independent of the loss landscape, which often leads to better generalisation in deep neural networks (Loshchilov and Hutter, 2019).

In [ ]:
# Weight decay of AdamW: each step shrinks the weights by a factor (1 - LEARNING_RATE * WEIGHT_DECAY)
WEIGHT_DECAY = 1e-1

In [ ]:
%%time
model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, metric=METRIC,
              patience=PATIENCE, verbose=10, live=5)
models['AdamW'], histories['AdamW'] = model, history
torch.save(model.state_dict(), weights_path('AdamW'))

In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, top=10, normalize=True,
                      title='Confusion matrix (validation set)')

#### **Experiment 6: Early Stopping, AdamW and Dropout**

The two regularisers together: decoupled weight decay and dropout.

In [ ]:
%%time
model = make_model(DROPOUT_RATE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, metric=METRIC,
              patience=PATIENCE, verbose=10, live=5)
models['AdamW + dropout'], histories['AdamW + dropout'] = model, history
torch.save(model.state_dict(), weights_path('AdamW + dropout'))

In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, top=10, normalize=True,
                      title='Confusion matrix (validation set)')

### 📈 **Results Comparison**

Validation curves of all the experiments, smoothed to show the trend, and the validation F1 score of each final model: the restored best weights with early stopping, the last epoch for the baseline. The smallest topics have only 2 or 3 validation articles, and a single one of them changing prediction can move the macro F1 score by more than a point, so small differences are noise; Lecture 3 measures this properly with cross-validation.

In [ ]:
# Accuracy and macro F1 score of each final model on the validation set
val_scores = {}
for name, m in models.items():
    y_true, y_pred = predict(m, val_loader, device)
    val_scores[name] = score(y_true, y_pred, METRIC)
    print(f"{name:16s} accuracy {accuracy_score(y_true, y_pred):.4f} | F1 score (macro) {val_scores[name]:.4f}")
best_name = max(val_scores, key=val_scores.get)

fig, axes = plt.subplots(1, 3, figsize=(20, 4.8), gridspec_kw={'width_ratios': [3, 3, 2]})

# Validation curves, one colour per experiment, smoothed over 5 epochs to show the trend
for colour, (name, history) in zip(PALETTE, histories.items()):
    for ax, key in zip(axes[:2], ['val_loss', f'val_{METRIC}']):
        trend = pd.Series(history[key]).rolling(5, min_periods=1).mean()
        ax.plot(range(1, len(trend) + 1), trend, color=colour, linewidth=1.5, label=name)
axes[0].set(title='Validation loss (5-epoch average)', xlabel='Epoch')
axes[1].set(title='Validation F1 score (macro, 5-epoch average)', xlabel='Epoch')
axes[0].legend(loc='upper left')

# Final validation score of each experiment
names = list(val_scores)
axes[2].barh(names, [val_scores[n] for n in names], color=PALETTE[0], height=0.6)
for i, name in enumerate(names):
    axes[2].text(val_scores[name] + 0.003, i, f"{val_scores[name]:.3f}", va='center')
axes[2].invert_yaxis()
axes[2].set(title='Final model on the validation set', xlabel='F1 score (macro)',
            xlim=(min(val_scores.values()) - 0.05, max(val_scores.values()) + 0.04))
axes[2].grid(axis='y', visible=False)

plt.tight_layout()
plt.show()
print(f"Best experiment on the validation set: {best_name} (F1 score {val_scores[best_name]:.4f})")

#### **Weights of the Final Models**

Regularisation acts on the weights, so it is worth looking at them: the distribution of the weights of each layer in every final model (biases left out, counts on a log scale), and their root mean square. Compare how far each technique pulls the weights towards zero: the L2 penalty and weight decay shrink them directly, early stopping indirectly, by training for fewer epochs than the baseline, while dropout regularises through noise rather than size (compare it with early stopping alone). Look also at the output layer of the baseline: large weights there mean large logits, that is, over-confident predictions, which make the validation loss grow even while the F1 score holds.

In [ ]:
# Weights of each linear layer in every final model, biases left out
weights = {name: [module.weight.detach().cpu().flatten().numpy() for module in model.modules()
                  if isinstance(module, nn.Linear)] for name, model in models.items()}
num_layers = len(next(iter(weights.values())))

fig, axes = plt.subplots(1, num_layers, figsize=(4.5 * num_layers, 4.8))
for i, ax in enumerate(axes):
    # Same bins for all the experiments, covering 99.9% of the weights of this layer
    limit = max(np.percentile(np.abs(layer_weights[i]), 99.9) for layer_weights in weights.values())
    bins = np.linspace(-limit, limit, 61)
    for colour, (name, layer_weights) in zip(PALETTE, weights.items()):
        ax.hist(layer_weights[i], bins=bins, histtype='step', color=colour, linewidth=1.5, label=name)
    ax.set(title=f'Layer {i + 1}' + (' (output)' if i == num_layers - 1 else ''), xlabel='Weight', yscale='log')
axes[0].set_ylabel('Number of weights')
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper center', ncol=len(weights), frameon=False)
plt.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

# Root mean square of the weights: one number per layer and experiment
pd.DataFrame({name: [np.sqrt(np.mean(w ** 2)) for w in layer_weights] for name, layer_weights in weights.items()},
             index=[f'Layer {i + 1}' for i in range(num_layers)]).T.round(4)

### 🕹️ **Inference**

In [ ]:
# The best model on the validation set, evaluated once on the test set
best_model = models[best_name]
y_true, y_pred = predict(best_model, test_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, top=10, normalize=True,
                      title=f'Confusion matrix (test set, {best_name})')

#### **Save, Load and Classify New Text**

A trained model is usually saved to a file and loaded later, in another notebook or in an application. We save the weights of the best model, load them into a new network of the same architecture, and ask it about two texts of our own: a typical newswire, and a harder one where two topics are plausible. Look at how the probability is shared among the topics, not only at the winner. The texts go through the same encoding as the articles (lower case, split into words, multi-hot over the same 2,000-word vocabulary), and the words outside the vocabulary are ignored.

In [ ]:
# Save the weights of the best model to a file
torch.save(best_model.state_dict(), weights_path('best model'))
print(f"Saved {weights_path('best model')} ({os.path.getsize(weights_path('best model')) / 1e6:.1f} MB)")

# Load them into a new network: the architecture must match, dropout layers included
dropout_rate = next((module.p for module in best_model.modules() if isinstance(module, nn.Dropout)), 0.0)
loaded_model = make_model(dropout_rate)
loaded_model.load_state_dict(torch.load(weights_path('best model'), map_location=device))
loaded_model.eval()
print(f"Loaded the weights of '{best_name}' into a new network")

In [ ]:
# From word to rank, the inverse of index_word
word_index = {word: rank for rank, word in index_word.items()}


def encode(text):
    """Multi-hot vector of a text over the vocabulary, as for the articles; also return the words left out."""
    words = re.findall(r"[a-z0-9']+", text.lower())
    ranks = [word_index.get(word, VOCAB_SIZE + 1) for word in words]
    x = np.zeros(VOCAB_SIZE, dtype=np.float32)
    x[[rank - 1 for rank in ranks if rank <= VOCAB_SIZE]] = 1.0
    ignored = [word for word, rank in zip(words, ranks) if rank > VOCAB_SIZE]
    return x, ignored

In [ ]:
# A typical newswire, and a harder text that could belong to two topics
texts = {
    'Typical newswire': "Saudi Arabia said OPEC members agreed to cut crude oil output by one mln barrels per day "
                        "to support prices",
    'Harder: trade or grain?': "Japan and the United States agreed to cut tariffs on rice, the trade minister said",
}

fig, axes = plt.subplots(1, len(texts), figsize=(16, 3.8))
for ax, (title, text) in zip(axes, texts.items()):
    x, ignored = encode(text)
    with torch.no_grad():
        logits = loaded_model(torch.from_numpy(x).unsqueeze(0).to(device))
    probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()

    # The five most likely topics, the most likely at the top
    top = np.argsort(probabilities)[::-1][:5]
    ax.barh([class_names[k] for k in top][::-1], probabilities[top][::-1], color=PALETTE[0], height=0.6)
    ax.set(title=title, xlabel='Probability', xlim=(0, 1))
    ax.grid(axis='y', visible=False)
    print(f"{title}: \"{text}\"")
    print(f"Words outside the vocabulary: {', '.join(ignored) or 'none'}\n")
plt.tight_layout()
plt.show()

### 📜 **Homework Exercise**
Reuters is a large text dataset with thousands of sparse features. Repeat the analysis on the Yeast dataset, a small tabular one: predict in which part of the cell a protein is located from 8 numerical scores, among 10 sites whose sizes range from 463 proteins down to 5. Compare accuracy with the macro F1 score, and check whether the regularisation techniques of this lecture help here too. Pay attention to the smallest classes when you split the data!

<img src="https://upload.wikimedia.org/wikipedia/commons/4/48/Simple_diagram_of_yeast_cell_%28en%29.svg" width="500">

```
# Data loading
DATASET_NAME = "yeast.zip"
DATASET_ID = "1lTQkbiz1ed7A3Z6X2JgQd2BNIDsTS3Qn"
if not os.path.exists(DATASET_NAME):
    !gdown -q {DATASET_ID} -O {DATASET_NAME}
    !unzip -q -o {DATASET_NAME}

# yeast.data has no header: protein name, 8 scores and localisation site, separated by spaces
columns = ['Sequence Name', 'mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc', 'Localisation Site']
dataset = pd.read_csv('yeast.data', sep=r'\s+', header=None, names=columns)
```

#  
<img src="https://airlab.deib.polimi.it/wp-content/uploads/2019/07/airlab-logo-new_cropped.png" width="350">

##### Connect with us:
- <img src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" width="14"> **LinkedIn:**  [AIRLab Polimi](https://www.linkedin.com/company/airlab-polimi/)
- <img src="https://upload.wikimedia.org/wikipedia/commons/9/95/Instagram_logo_2022.svg" width="14"> **Instagram:** [airlab_polimi](https://www.instagram.com/airlab_polimi/)

##### Contributors:
- **Eugenio Lomurno**: eugenio.lomurno@polimi.it
- **Alberto Archetti**: alberto.archetti@polimi.it
- **Roberto Basla**: roberto.basla@polimi.it
- **Carlo Sgaravatti**: carlo.sgaravatti@polimi.it

```
   Copyright 2026 Eugenio Lomurno, Alberto Archetti, Roberto Basla, Carlo Sgaravatti

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```
